In [1]:
import pandas as pd  
 
dvf = pd.read_csv("../data/raw/dvf_69.csv", sep= ',', dtype={ "numero_volume": "string", "lot1_numero": "string"})
# ============================================================
# Filtre nature_mutation et type_local
dvf = dvf[(dvf["nature_mutation"] == "Vente") & (dvf["type_local"].isin(["Maison", "Appartement"]))]
            #  & (dvf["surface_reelle_bati"] > 0) & (dvf["valeur_fonciere"] > 0) ]
            
print("\nTypes de variables :"); display(dvf.dtypes.value_counts())
print("\nDoublons :"); print(dvf.duplicated().sum())
display('Dimension base maison/ appat :', dvf.shape)
# ============================================================
# NETTOYAGE - SUPPRESSION DES DOUBLONS STRICTS
nb_avant = len(dvf) 
dvf = dvf.drop_duplicates().copy()
nb_apres = len(dvf) 

print("Nombre de lignes avant :", nb_avant)
print("Nombre de lignes après :", nb_apres)
print("Nombre de lignes supprimées :", nb_avant - nb_apres)

# Contrôle
print( "Doublons stricts restants :",  dvf.duplicated().sum())

# Mutation avec plusieurs logements
display("Mutation avec plusieurs logements: ",dvf[dvf["id_mutation"] == "2021-1131987"])
# Mutation multilignes liée au foncier
display(" Mutation multilignes liée au foncier : " ,dvf[dvf["id_mutation"] == "2021-1132368"])



Types de variables :


float64    21
str        13
int64       4
string      2
Name: count, dtype: int64


Doublons :
7200


'Dimension base maison/ appat :'

(147641, 40)

Nombre de lignes avant : 147641
Nombre de lignes après : 140441
Nombre de lignes supprimées : 7200
Doublons stricts restants : 0


'Mutation avec plusieurs logements: '

,id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,...,type_local,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude
466,2021-1131987,2021-01-15,3,Vente,395000.0,1.0,NaN,RUE LOUIS BURDIN,0016,69580.0,...,Maison,28.0,1.0,S,sols,NaN,NaN,413.0,4.877784,45.833627
467,2021-1131987,2021-01-15,3,Vente,395000.0,1.0,NaN,RUE LOUIS BURDIN,0016,69580.0,...,Maison,82.0,5.0,S,sols,NaN,NaN,413.0,4.877784,45.833627


' Mutation multilignes liée au foncier : '

,id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,...,type_local,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude
1117,2021-1132368,2021-02-16,1,Vente,455500.0,3.0,NaN,GORGES D ENFER,B022,69650.0,...,Maison,95.0,4.0,S,sols,NaN,NaN,500.0,4.816413,45.878242
1118,2021-1132368,2021-02-16,1,Vente,455500.0,3.0,NaN,GORGES D ENFER,B022,69650.0,...,Maison,95.0,4.0,T,terres,NaN,NaN,1700.0,4.816413,45.878242
1119,2021-1132368,2021-02-16,1,Vente,455500.0,3.0,NaN,GORGES D ENFER,B022,69650.0,...,Maison,95.0,4.0,BT,taillis simples,NaN,NaN,11167.0,4.816413,45.878242


In [2]:
# ============================================================
# PREPROCESSING - nombre_pieces_principales

# 1. Vérifier les NA et la distribution
print("Taux de NA (%) :", round(dvf["nombre_pieces_principales"].isna().mean() * 100, 2))
print(dvf["nombre_pieces_principales"].value_counts(dropna=False).sort_index())

# 2. Comparer selon le type de logement
display(dvf.groupby("type_local")["nombre_pieces_principales"].describe())

# 3. Vérifier les valeurs nulles ou incohérentes
print("Nombre de pièces = 0 :", (dvf["nombre_pieces_principales"] == 0).sum())
print("Nombre de pièces < 0 :", (dvf["nombre_pieces_principales"] < 0).sum())

Taux de NA (%) : 0.01
nombre_pieces_principales
0.0       234
1.0     16228
2.0     26096
3.0     36706
4.0     34709
5.0     17257
6.0      5877
7.0      2058
8.0       782
9.0       222
10.0      119
11.0       53
12.0       43
13.0       11
14.0        7
15.0        2
16.0        3
17.0       10
18.0        1
19.0        1
21.0        1
22.0        1
23.0        1
28.0        1
32.0        1
34.0        3
NaN        14
Name: count, dtype: int64


,count,mean,std,min,25%,50%,75%,max
type_local,,,,,,,,
Appartement,98931.0,2.817327,1.228929,0.0,2.0,3.0,4.0,34.0
Maison,41496.0,4.480022,1.472153,0.0,4.0,4.0,5.0,34.0


Nombre de pièces = 0 : 234
Nombre de pièces < 0 : 0


In [3]:
# ============================================================
# PREPROCESSING - id_mutation

print("Nombre de lignes :", len(dvf))
print("Taux de NA (%) :", round(dvf["id_mutation"].isna().mean() * 100, 2))
print("Cardinalité :", dvf["id_mutation"].nunique())

# Vérifier les mutations présentes sur plusieurs lignes
nb_lignes_par_mutation = dvf.groupby("id_mutation").size()
print("Part des mutations multilignes (%) :", round((nb_lignes_par_mutation > 1).mean() * 100, 2))

# ============================================================
# 1) Exclure les mutations multi-logements
# 2) Consolider les mutations répétées liées au terrain

# ------------------------------------------------------------
# Identifier les mutations avec plusieurs profils de logements
profils_logement = (dvf[["id_mutation", "type_local", "surface_reelle_bati", "nombre_pieces_principales"]].drop_duplicates())

nb_profils = (profils_logement.groupby("id_mutation").size())
ids_multi_logements = nb_profils[nb_profils > 1].index

print("Mutations multi-logements :", len(ids_multi_logements))
print("Lignes concernées :", dvf["id_mutation"].isin(ids_multi_logements).sum())

# Exclusion car valeur_fonciere = prix global de la mutation
dvf_clean = dvf[~dvf["id_mutation"].isin(ids_multi_logements)].copy()

# ------------------------------------------------------------
# Consolider les mutations restantes liées au terrain

# Une seule occurrence de chaque composante foncière
terrains_uniques = (dvf_clean[["id_mutation", "id_parcelle", "nature_culture", "nature_culture_speciale", "surface_terrain"]].drop_duplicates())

# Surface totale du terrain par mutation
surface_terrain_totale = (terrains_uniques.groupby("id_mutation")["surface_terrain"].sum(min_count=1).rename("surface_terrain_totale"))

# Ajouter la surface consolidée
dvf_clean = dvf_clean.merge(surface_terrain_totale, on="id_mutation", how="left")

# Une ligne par mutation
dvf_clean = (dvf_clean.drop_duplicates(subset=["id_mutation"], keep="first").copy())

# Remplacer l'ancienne surface terrain
dvf_clean["surface_terrain"] = dvf_clean["surface_terrain_totale"]
dvf_clean.drop(columns=["surface_terrain_totale"], inplace=True)

# ------------------------------------------------------------
# Contrôle final

print("Nombre de lignes finales :", len(dvf_clean))
print("Mutations uniques :", dvf_clean["id_mutation"].nunique())
print("Mutations encore multilignes :", (dvf_clean.groupby("id_mutation").size() > 1).sum())

print("dvf :", dvf.shape)
print("dvf_clean :", dvf_clean.shape)
dvf = dvf_clean.copy()

# Mutation avec plusieurs logements
display("Mutation avec plusieurs logements: ",dvf[dvf["id_mutation"] == "2021-1131987"])
# Mutation multilignes liée au foncier
display(" Mutation multilignes liée au foncier : " ,dvf[dvf["id_mutation"] == "2021-1132368"])


Nombre de lignes : 140441
Taux de NA (%) : 0.0
Cardinalité : 120349
Part des mutations multilignes (%) : 9.63
Mutations multi-logements : 5193
Lignes concernées : 18328
Nombre de lignes finales : 115156
Mutations uniques : 115156
Mutations encore multilignes : 0
dvf : (140441, 40)
dvf_clean : (115156, 40)


'Mutation avec plusieurs logements: '

,id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,...,type_local,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude


' Mutation multilignes liée au foncier : '

,id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,...,type_local,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude
460,2021-1132368,2021-02-16,1,Vente,455500.0,3.0,NaN,GORGES D ENFER,B022,69650.0,...,Maison,95.0,4.0,S,sols,NaN,NaN,13367.0,4.816413,45.878242


In [4]:
# ============================================================
# PREPROCESSING - numero_disposition

print("Nombre de lignes :", len(dvf))
print("Taux de NA (%) :", round(dvf["numero_disposition"].isna().mean() * 100, 2))
print("Cardinalité :", dvf["numero_disposition"].nunique())
print(dvf["numero_disposition"].describe())

dvf = dvf.drop(columns=["numero_disposition"])

Nombre de lignes : 115156
Taux de NA (%) : 0.0
Cardinalité : 11
count    115156.000000
mean          1.018575
std           0.160847
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          11.000000
Name: numero_disposition, dtype: float64


In [5]:
# ============================================================
# PREPROCESSING - nature_mutation 

dvf["nature_mutation"].value_counts(normalize=True).mul(100).round(2)
dvf = dvf.drop(columns=["nature_mutation"])

In [6]:
# ============================================================
# PREPROCESSING - surface_reelle_bati

#Vérifier les valeurs manquantes
print("Nombre de lignes :", len(dvf))
print("NA :", dvf["surface_reelle_bati"].isna().sum())
print("Taux de NA (%) :", round(dvf["surface_reelle_bati"].isna().mean() * 100, 2))

#Étudier la distribution et les valeurs extrêmes
print(dvf["surface_reelle_bati"].describe(percentiles=[.01, .50, .95, .99, .999]))
print("Asymétrie :", round(dvf["surface_reelle_bati"].skew(), 2))

#Comparer les valeurs extrêmes par type de logement
display(dvf.groupby("type_local")["surface_reelle_bati"].quantile([.01, .50, .95, .99, .999]).unstack())

#Nettoyer les valeurs manquantes
dvf = dvf.dropna(subset=["surface_reelle_bati"]).copy()
dvf = dvf[dvf["surface_reelle_bati"] > 0].copy()
# #Vérifier les valeurs incohérentes
print("Surfaces <= 0 :", (dvf["surface_reelle_bati"] <= 0).sum())

# #Contrôle final:
print("NA :", dvf["surface_reelle_bati"].isna().sum())
print("Nombre de lignes :", len(dvf))


Nombre de lignes : 115156
NA : 9
Taux de NA (%) : 0.01
count    115147.000000
mean         77.900623
std          38.825765
min           1.000000
1%           18.000000
50%          72.000000
95%         149.000000
99%         209.000000
99.9%       315.000000
max         786.000000
Name: surface_reelle_bati, dtype: float64
Asymétrie : 1.8


,0.010,0.500,0.950,0.990,0.999
type_local,,,,,
Appartement,17.0,64.0,113.0,155.0,231.000
Maison,40.0,101.0,189.0,258.0,386.492


Surfaces <= 0 : 0
NA : 0
Nombre de lignes : 115147


In [7]:
# ============================================================
# PREPROCESSING - adresse_numero
 
print("Taux de NA (%) :", round(dvf["adresse_numero"].isna().mean() * 100, 2))
print("Cardinalité :", dvf["adresse_numero"].nunique())
print(dvf["adresse_numero"].isnull().sum())

# dvf = dvf.drop(columns=["adresse_numero"])

Taux de NA (%) : 0.89
Cardinalité : 2428
1023


In [8]:
# ============================================================
# PREPROCESSING - adresse_suffixe
 
print("Taux de NA (%) :", round(dvf["adresse_suffixe"].isna().mean() * 100, 2))
print("Cardinalité :", dvf["adresse_suffixe"].nunique())
dvf = dvf.drop(columns=["adresse_suffixe"])

Taux de NA (%) : 93.51
Cardinalité : 25


In [9]:
# ============================================================
# PREPROCESSING -  adresse_nom_voie

# Conserver temporairement pour les contrôles géographiques.
# Ne pas utiliser dans la première baseline ML à cause de la forte cardinalité + Suppression dans le bloc de finalisation.
 
print("Taux de NA (%) :", round(dvf["adresse_nom_voie"].isna().mean() * 100, 2))
print(dvf["adresse_nom_voie"].nunique())
print(dvf["adresse_nom_voie"].value_counts().describe())

Taux de NA (%) : 0.0
11842
count    11842.000000
mean         9.723442
std         28.291868
min          1.000000
25%          1.000000
50%          2.000000
75%          7.000000
max        776.000000
Name: count, dtype: float64


In [10]:
# ============================================================
# PREPROCESSING - adresse_code_voie

# Conserver temporairement pour les contrôles géographiques.
# Ne pas utiliser dans la première baseline ML à cause de la forte cardinalité + Suppression dans le bloc de finalisation.

print("Taux de NA (%) :", round(dvf["adresse_code_voie"].isna().mean() * 100, 2))
print(dvf["adresse_code_voie"].nunique())
print(dvf["adresse_code_voie"].value_counts().describe())

display(dvf.groupby("adresse_code_voie")["adresse_nom_voie"].nunique().describe())

Taux de NA (%) : 0.0
2707
count    2707.000000
mean       42.536757
std        79.852254
min         1.000000
25%         4.000000
50%        15.000000
75%        43.000000
max       841.000000
Name: count, dtype: float64


count    2707.000000
mean        5.725896
std        10.623897
min         1.000000
25%         1.000000
50%         2.000000
75%         5.000000
max       102.000000
Name: adresse_nom_voie, dtype: float64

In [11]:
# ============================================================
# PREPROCESSING - code_postal (identifiant géographique catégoriel)

print("Taux de NA (%) :", round(dvf["code_postal"].isna().mean() * 100, 2))
print("Cardinalité :", dvf["code_postal"].nunique())

# Conversion en variable catégorielle
dvf["code_postal"] = dvf["code_postal"].astype("Int64").astype("string").str.zfill(5)

print("Type après conversion :", dvf["code_postal"].dtype)

Taux de NA (%) : 0.0
Cardinalité : 91
Type après conversion : string


In [12]:
# ============================================================
# PREPROCESSING - code_commune

print("Taux de NA (%) :", round(dvf["code_commune"].isna().mean() * 100, 2))
print("Cardinalité :", dvf["code_commune"].nunique())

# Conversion en variable catégorielle
dvf["code_commune"] = dvf["code_commune"].astype("string").str.zfill(5)

print("Type après conversion :", dvf["code_commune"].dtype)

Taux de NA (%) : 0.0
Cardinalité : 276


Type après conversion : string


In [13]:
# ============================================================
# PREPROCESSING - nom_commune (Conserver dans le dataset pour les contrôles + Redondante avec code_commune=> exclue des features avant modélisation)

print("Taux de NA (%) :", round(dvf["nom_commune"].isna().mean() * 100, 2))
print("Cardinalité :", dvf["nom_commune"].nunique())
print("Codes associés à plusieurs noms :", (dvf.groupby("code_commune")["nom_commune"].nunique() > 1).sum())

Taux de NA (%) : 0.0
Cardinalité : 276
Codes associés à plusieurs noms : 0


In [14]:
# ============================================================
# PREPROCESSING - code_departement

print("Taux de NA (%) :", round(dvf["code_departement"].isna().mean() * 100, 2))
print("Cardinalité :", dvf["code_departement"].nunique())

# 2. Vérifier les valeurs
print(dvf["code_departement"].value_counts(dropna=False))
dvf = dvf.drop(columns=["code_departement"])

Taux de NA (%) : 0.0
Cardinalité : 1
code_departement
69    115147
Name: count, dtype: int64


In [15]:
# ============================================================
# PREPROCESSING - ancien_code_commune / ancien_nom_commune / ancien_id_parcelle / numero_volume

print("NA ancien_code_commune (%) :", round(dvf["ancien_code_commune"].isna().mean() * 100, 2))
print("NA ancien_nom_commune (%) :", round(dvf["ancien_nom_commune"].isna().mean() * 100, 2))
print("NA ancien_id_parcelle (%) :", round(dvf["ancien_id_parcelle"].isna().mean() * 100, 2))
print("NA numero_volume (%) :", round(dvf["numero_volume"].isna().mean() * 100, 2))

# Supprimer les variables
dvf = dvf.drop(columns=["ancien_code_commune", "ancien_nom_commune","ancien_id_parcelle", "numero_volume"])

NA ancien_code_commune (%) : 100.0
NA ancien_nom_commune (%) : 100.0
NA ancien_id_parcelle (%) : 100.0
NA numero_volume (%) : 100.0


In [16]:
# ============================================================
# PREPROCESSING - id_parcelle (Conserver pendant le preprocessing + Supprimer uniquement lors de la création du dataset final ML)

print("Taux de NA (%) :", round(dvf["id_parcelle"].isna().mean() * 100, 2))
print("Cardinalité :", dvf["id_parcelle"].nunique())
print(dvf["id_parcelle"].value_counts().describe())

#  Vérifier combien de parcelles apparaissent plusieurs fois
print("Parcelles présentes plusieurs fois :", (dvf["id_parcelle"].value_counts() > 1).sum())

Taux de NA (%) : 0.0
Cardinalité : 46123
count    46123.000000
mean         2.496520
std          4.800757
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max        179.000000
Name: count, dtype: float64
Parcelles présentes plusieurs fois : 13503


In [17]:
# ============================================================
# PREPROCESSING - lot1_numero

print("Taux de NA (%) :", round(dvf["lot1_numero"].isna().mean() * 100, 2))
print("Cardinalité :", dvf["lot1_numero"].nunique())
display(pd.crosstab(dvf["type_local"], dvf["lot1_numero"].isna(), normalize="index") * 100)
print(dvf["lot1_numero"].value_counts().describe())
print(dvf["lot1_numero"].isnull().sum())

Taux de NA (%) : 26.38
Cardinalité : 2313


lot1_numero,False,True
type_local,,
Appartement,99.343647,0.656353
Maison,6.664581,93.335419


count        2313.0
mean      36.649805
std      168.179755
min             1.0
25%             1.0
50%             2.0
75%             7.0
max          2563.0
Name: count, dtype: double[pyarrow]
30376


In [18]:
# ============================================================
# PREPROCESSING - lot2 à lot5

cols_lots = ["lot2_numero", "lot2_surface_carrez", "lot3_numero", "lot3_surface_carrez", "lot4_numero", "lot4_surface_carrez", "lot5_numero", "lot5_surface_carrez"]
print((dvf[cols_lots].isna().mean() * 100).round(2))

print(dvf[cols_lots].isnull().sum())
# Vérifier le taux de valeurs renseignées selon le type de logement
display(dvf.groupby("type_local")[cols_lots].agg(lambda x: round(x.notna().mean() * 100, 2)))

dvf = dvf.drop(columns=cols_lots)

lot2_numero            64.56
lot2_surface_carrez    84.94
lot3_numero            96.80
lot3_surface_carrez    99.29
lot4_numero            99.19
lot4_surface_carrez    99.85
lot5_numero            99.72
lot5_surface_carrez    99.96
dtype: float64
lot2_numero             74340
lot2_surface_carrez     97803
lot3_numero            111467
lot3_surface_carrez    114334
lot4_numero            114214
lot4_surface_carrez    114969
lot5_numero            114819
lot5_surface_carrez    115102
dtype: int64


,lot2_numero,lot2_surface_carrez,lot3_numero,lot3_surface_carrez,lot4_numero,lot4_surface_carrez,lot5_numero,lot5_surface_carrez
type_local,,,,,,,,
Appartement,48.77,20.77,4.33,0.95,1.10,0.21,0.38,0.05
Maison,0.74,0.19,0.24,0.07,0.07,0.01,0.04,0.01


In [19]:
# ============================================================
# PREPROCESSING - lot1_surface_carrez

print("Taux de NA (%) :", round(dvf["lot1_surface_carrez"].isna().mean() * 100, 2))
print(dvf.groupby("type_local")["lot1_surface_carrez"].apply(lambda x: round(x.isna().mean() * 100, 2)))

# # Étudier la distribution
print(dvf["lot1_surface_carrez"].describe(percentiles=[.01, .50, .95, .99, .999]))

# 3. Corrélation avec la surface bâtie
corr = dvf["lot1_surface_carrez"].corr(dvf["surface_reelle_bati"])
print("Corrélation avec surface_reelle_bati :", round(corr, 3))

Taux de NA (%) : 58.17
type_local
Appartement    43.9
Maison         95.3
Name: lot1_surface_carrez, dtype: float64
count    48168.000000
mean        63.438235
std         52.023311
min          1.320000
1%          16.493400
50%         62.290000
95%        111.606500
99%        156.069700
99.9%      237.911350
max       7570.000000
Name: lot1_surface_carrez, dtype: float64
Corrélation avec surface_reelle_bati : 0.525


In [20]:
# ============================================================
# PREPROCESSING - nombre_lots

# 1. Vérifier les NA et la distribution
print("Taux de NA (%) :", round(dvf["nombre_lots"].isna().mean() * 100, 2))
print(dvf["nombre_lots"].describe())
print(dvf["nombre_lots"].value_counts(normalize=True).sort_index().head(10) * 100)

# 2. Comparer selon le type de logement
display(dvf.groupby("type_local")["nombre_lots"].describe())

# 3. Feature engineering : présence d'au moins un lot
# dvf["presence_lot"] = (dvf["nombre_lots"] > 0).astype(int)


Taux de NA (%) : 0.0
count    115147.000000
mean          1.135705
std           0.884589
min           0.000000
25%           0.000000
50%           1.000000
75%           2.000000
max          22.000000
Name: nombre_lots, dtype: float64
nombre_lots
0    26.380192
1    38.180760
2    32.243133
3     2.385646
4     0.525415
5     0.159796
6     0.080766
7     0.028659
8     0.006079
9     0.004342
Name: proportion, dtype: float64


,count,mean,std,min,25%,50%,75%,max
type_local,,,,,,,,
Appartement,83187.0,1.542152,0.668606,0.0,1.0,1.0,2.0,22.0
Maison,31960.0,0.077785,0.326295,0.0,0.0,0.0,0.0,9.0


In [21]:
# ============================================================
# PREPROCESSING - code_type_local

# Vérifier les NA et la correspondance avec type_local
print("Taux de NA (%) :", round(dvf["code_type_local"].isna().mean() * 100, 2))
display(pd.crosstab(dvf["code_type_local"], dvf["type_local"], dropna=False))

dvf = dvf.drop(columns=["code_type_local"])

Taux de NA (%) : 0.0


type_local,Appartement,Maison
code_type_local,,
1.0,0,31960
2.0,83187,0


In [22]:
# ============================================================
# PREPROCESSING - type_local

print("Taux de NA (%) :", round(dvf["type_local"].isna().mean() * 100, 2))
print(dvf["type_local"].value_counts(dropna=False))

# Vérifier la répartition Maison / Appartement
print(dvf["type_local"].value_counts(normalize=True).mul(100).round(2))

# Conserver comme variable catégorielle
dvf["type_local"] = dvf["type_local"].astype("string")

Taux de NA (%) : 0.0
type_local
Appartement    83187
Maison         31960
Name: count, dtype: int64
type_local
Appartement    72.24
Maison         27.76
Name: proportion, dtype: float64


In [23]:
# ============================================================
# PREPROCESSING - nombre_pieces_principales

print("Taux de NA (%) :", round(dvf["nombre_pieces_principales"].isna().mean() * 100, 2))
print(dvf["nombre_pieces_principales"].value_counts(dropna=False).sort_index())
display(dvf.groupby("type_local")["nombre_pieces_principales"].describe())

print("Nombre de pièces = 0 :", (dvf["nombre_pieces_principales"] == 0).sum())
print("Nombre de pièces < 0 :", (dvf["nombre_pieces_principales"] < 0).sum())

dvf.loc[dvf["nombre_pieces_principales"] == 0, "nombre_pieces_principales"] = pd.NA

Taux de NA (%) : 0.0
nombre_pieces_principales
0.0       114
1.0     12099
2.0     20962
3.0     31202
4.0     29861
5.0     14255
6.0      4432
7.0      1430
8.0       518
9.0       135
10.0       64
11.0       32
12.0       24
13.0        5
14.0        5
15.0        1
16.0        1
17.0        3
23.0        1
32.0        1
34.0        2
Name: count, dtype: int64


,count,mean,std,min,25%,50%,75%,max
type_local,,,,,,,,
Appartement,83187.0,2.875798,1.208858,0.0,2.0,3.0,4.0,34.0
Maison,31960.0,4.464956,1.355314,0.0,4.0,4.0,5.0,34.0


Nombre de pièces = 0 : 114
Nombre de pièces < 0 : 0


In [24]:
# ============================================================
# PREPROCESSING - code_nature_culture / nature_culture

print("Taux de NA (%) :", round(dvf["nature_culture"].isna().mean() * 100, 2))
print(dvf.groupby("type_local")["nature_culture"].apply(lambda x: round(x.isna().mean() * 100, 2)))
print(dvf["nature_culture"].isnull().sum())
print(dvf["nature_culture"].value_counts(dropna=False))

# Vérifier la redondance code + libellé
display(pd.crosstab(dvf["code_nature_culture"], dvf["nature_culture"], dropna=False))

dvf = dvf.drop(columns=["code_nature_culture"])

Taux de NA (%) : 73.97
type_local
Appartement    99.17
Maison          8.36
Name: nature_culture, dtype: float64
85171
nature_culture
NaN                    85171
sols                   26176
terrains d'agrément     2211
jardins                  763
prés                     256
terrains a bâtir         234
terres                   205
taillis simples           45
vergers                   35
landes                    16
vignes                    12
pâtures                    8
bois                       5
taillis sous futaie        4
eaux                       3
futaies résineuses         2
futaies feuillues          1
Name: count, dtype: int64


nature_culture,bois,eaux,futaies feuillues,futaies résineuses,jardins,landes,prés,pâtures,sols,taillis simples,taillis sous futaie,terrains a bâtir,terrains d'agrément,terres,vergers,vignes,NaN
code_nature_culture,,,,,,,,,,,,,,,,,
AB,0,0,0,0,0,0,0,0,0,0,0,234,0,0,0,0,0
AG,0,0,0,0,0,0,0,0,0,0,0,0,2211,0,0,0,0
B,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
BF,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
BR,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0
BS,0,0,0,0,0,0,0,0,0,0,4,0,0,0,0,0,0
BT,0,0,0,0,0,0,0,0,0,45,0,0,0,0,0,0,0
E,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
J,0,0,0,0,763,0,0,0,0,0,0,0,0,0,0,0,0


In [25]:
# ============================================================
# PREPROCESSING - code_nature_culture_speciale / nature_culture_speciale

print("Taux NA code (%) :", round(dvf["code_nature_culture_speciale"].isna().mean() * 100, 2))
print("Taux NA libellé (%) :", round(dvf["nature_culture_speciale"].isna().mean() * 100, 2))
print("Cardinalité :", dvf["nature_culture_speciale"].nunique())

dvf = dvf.drop(columns=["code_nature_culture_speciale", "nature_culture_speciale"])


Taux NA code (%) : 98.2
Taux NA libellé (%) : 98.2
Cardinalité : 9


In [26]:
# ============================================================
# PREPROCESSING - surface_terrain

print("Taux de NA (%) :", round(dvf["surface_terrain"].isna().mean() * 100, 2))
print(dvf.groupby("type_local")["surface_terrain"].apply(lambda x: round(x.isna().mean() * 100, 2)))
display(dvf.groupby("type_local")["surface_terrain"].agg(["count", "median", "mean"]))

# Vérifier les valeurs incohérentes
print("Surfaces <= 0 :", (dvf["surface_terrain"] <= 0).sum())

# Etudier les valeurs extrêmes
print(dvf["surface_terrain"].describe(percentiles=[.01, .50, .95, .99, .999]))

# Examiner les plus grandes surfaces
display(dvf.nlargest(10, "surface_terrain")[["id_mutation", "type_local", "surface_terrain", "surface_reelle_bati", "valeur_fonciere"]])


Taux de NA (%) : 73.93
type_local
Appartement    99.14
Maison          8.32
Name: surface_terrain, dtype: float64


,count,median,mean
type_local,,,
Appartement,717,187.0,677.845188
Maison,29302,592.0,859.443860


Surfaces <= 0 : 0


count     30019.000000
mean        855.106399
std        1860.232696
min           2.000000
1%           37.000000
50%         583.000000
95%        2250.000000
99%        5245.820000
99.9%     19566.772000
max      127748.000000
Name: surface_terrain, dtype: float64


,id_mutation,type_local,surface_terrain,surface_reelle_bati,valeur_fonciere
62310,2023-901684,Maison,127748.0,109.0,198575.0
27302,2021-1170816,Maison,119221.0,95.0,125000.0
39128,2022-1116056,Maison,115507.0,68.0,135630.0
32731,2022-1106906,Maison,57206.0,36.0,95000.0
106346,2025-868992,Maison,57206.0,36.0,67500.0
36703,2022-1112551,Maison,46142.0,80.0,92000.0
82681,2024-815246,Maison,45545.0,140.0,280000.0
67449,2023-908990,Maison,42591.0,150.0,350000.0
61618,2023-900662,Maison,34023.0,54.0,165000.0
109363,2025-872971,Maison,33940.0,200.0,275000.0


In [27]:
# ============================================================
# PREPROCESSING - longitude / latitude

import numpy as np
from geopy.distance import geodesic

print("NA longitude (%) :", round(dvf["longitude"].isna().mean() * 100, 2))
print("NA latitude (%) :", round(dvf["latitude"].isna().mean() * 100, 2))
display(pd.crosstab(dvf["latitude"].isna(), dvf["longitude"].isna()))


NA longitude (%) : 1.7
NA latitude (%) : 1.7


longitude,False,True
latitude,,
False,113190,0
True,0,1957


In [28]:
# ============================================================
# PREPROCESSING - valeur_fonciere

# Vérifier les valeurs manquantes
print("NA :", dvf["valeur_fonciere"].isna().sum())
print("Taux de NA (%) :", round(dvf["valeur_fonciere"].isna().mean() * 100, 2))

# Supprimer les valeurs manquantes de la cible
dvf = dvf.dropna(subset=["valeur_fonciere", "surface_reelle_bati"]).copy()

# Conserver uniquement les surfaces bâties positives
dvf = dvf[dvf["surface_reelle_bati"] > 0].copy()

# Étudier la distribution de valeur_fonciere
print(dvf["valeur_fonciere"].describe(percentiles=[.01, .50, .95, .99, .999]))
print("Asymétrie :", round(dvf["valeur_fonciere"].skew(), 2))

# Comparer selon le type de logement
display(dvf.groupby("type_local")["valeur_fonciere"].quantile([.01, .50, .99, .999]).unstack())
 

NA : 41
Taux de NA (%) : 0.04
count    1.151060e+05
mean     3.061622e+05
std      2.577817e+05
min      5.000000e-01
1%       4.556845e+04
50%      2.500000e+05
95%      6.900000e+05
99%      1.155347e+06
99.9%    2.510673e+06
max      1.750000e+07
Name: valeur_fonciere, dtype: float64
Asymétrie : 12.49


,0.010,0.500,0.990,0.999
type_local,,,,
Appartement,50000.0,218100.0,840000.0,1768953.5
Maison,39215.0,380000.0,1563871.0,3571425.0


In [29]:
# ============================================================
#  Features

import sys
import os
sys.path.append(os.path.abspath(".."))
from src.features.build_features import creer_features, detecter_prix_extremes, finaliser_datasets

# Création des features
dvf_initial = creer_features(dvf)
display(dvf_initial.head())

# Traitement des prix extrêmes
dvf_nettoye, valeurs_extremes, tableau_seuils = detecter_prix_extremes(dvf_initial)

display(tableau_seuils)
print("Nombre de lignes avant filtrage :", len(dvf))
print("Nombre de lignes après filtrage :", len(dvf_nettoye))
print("Nombre de valeurs extrêmes :", len(valeurs_extremes))

assert len(dvf) == len(dvf_initial), "Le nombre de lignes a changé."

# ============================================================
# La surface_terrain est absente pour 99,3 % des appartements mais seulement 8,3 % des maisons.
# Pour les appartements, les valeurs manquantes sont remplacées par 0 car ils ne disposent généralement
# pas de terrain individuel. Pour les maisons, les valeurs manquantes sont conservées et seront imputées
# lors du preprocessing. Une variable terrain_renseigne est ajoutée pour indiquer si l'information
# sur la surface du terrain était disponible.

# # Indicateur AVANT remplacement
# dvf_nettoye["terrain_renseigne"] = (
#     dvf_nettoye["surface_terrain"].notna().astype(int)
# )

# # Appartements : NA -> 0
# masque_appartements = (
#     (dvf_nettoye["type_local"] == "Appartement") &
#     (dvf_nettoye["surface_terrain"].isna())
# )

# dvf_nettoye.loc[
#     masque_appartements,
#     "surface_terrain"
# ] = 0


# # Finalisation et export des deux datasets
dvf_initial_final, dvf_nettoye_final = finaliser_datasets(dvf_initial, dvf_nettoye)

taux_na = pd.DataFrame({
    "Taux de NA (%) - initial": (dvf_initial_final.isna().mean() * 100).round(2),
    "Taux de NA (%) - nettoyé": (dvf_nettoye_final.isna().mean() * 100).round(2)
})
display(taux_na)

# ============================================================
# Analyse comparative avant / après traitement

comparaison_prix = pd.DataFrame({
    "Avant traitement": [
        len(dvf_initial_final),
        dvf_initial_final["prix_m2"].mean(),
        dvf_initial_final["prix_m2"].median(),
        dvf_initial_final["prix_m2"].std(),
        dvf_initial_final["prix_m2"].min(),
        dvf_initial_final["prix_m2"].max()
    ],
    "Après traitement": [
        len(dvf_nettoye_final),
        dvf_nettoye_final["prix_m2"].mean(),
        dvf_nettoye_final["prix_m2"].median(),
        dvf_nettoye_final["prix_m2"].std(),
        dvf_nettoye_final["prix_m2"].min(),
        dvf_nettoye_final["prix_m2"].max()
    ]
}, index=[
    "Nombre d'observations",
    "Prix moyen au m²",
    "Prix médian au m²",
    "Écart-type",
    "Prix minimum au m²",
    "Prix maximum au m²"
])

display(comparaison_prix.round(2))

print("\nLes deux datasets sont prêts pour la modélisation.")
print("DVF initial avec prix extrêmes :", dvf_initial_final.shape)
print("DVF nettoyé sans prix extrêmes :", dvf_nettoye_final.shape)
print("Nombre de prix extrêmes retirés :", len(dvf_initial_final) - len(dvf_nettoye_final))
print("\nFichiers créés :")
print("- ../data/processed/dvf_initial.csv")
print("- ../data/processed/dvf_nettoye.csv")


,id_mutation,date_mutation,valeur_fonciere,adresse_numero,adresse_nom_voie,adresse_code_voie,code_postal,code_commune,nom_commune,id_parcelle,...,nature_culture,surface_terrain,longitude,latitude,annee_mutation,mois_mutation,prix_m2,surface_par_piece,log_surface,distance_lyon_km
0,2021-1131719,2021-01-05,375000.0,46.0,RUE DOC ALBERIC PONT,2193,69005,69385,Lyon 5e Arrondissement,69385000BO0189,...,NaN,NaN,4.797315,45.754569,2021,1,3472.222222,21.600000,4.691348,3.164923
1,2021-1131721,2021-01-04,203050.0,10.0,AV DE MENIVAL,4720,69005,69385,Lyon 5e Arrondissement,69385000BC0024,...,NaN,NaN,4.786061,45.757444,2021,1,2446.385542,16.600000,4.430817,3.929888
2,2021-1131723,2021-01-04,140000.0,250.0,RUE DES ERABLES,2530,69009,69389,Lyon 9e Arrondissement,69389000AS0091,...,NaN,NaN,4.795657,45.782647,2021,1,1917.808219,24.333333,4.304065,3.741073
3,2021-1131724,2021-01-04,174100.0,54.0,RUE PIERRE AUDRY,5450,69009,69389,Lyon 9e Arrondissement,69389000CD0032,...,NaN,NaN,4.806255,45.763372,2021,1,2523.188406,23.000000,4.248495,2.291661
4,2021-1131726,2021-01-07,470250.0,21.0,RUE DES CEDRES,0022,69650,69207,Saint-Germain-au-Mont-d'Or,69207000AO0092,...,sols,705.0,4.807607,45.879440,2021,1,3886.363636,24.200000,4.804021,13.015311


,Type,Borne basse (€/m²),Borne haute (€/m²),Nombre total,Valeurs extrêmes
0,Appartement,1334.868278,10681.433087,83162,2315
1,Maison,1148.531697,11437.473734,31944,1902


Nombre de lignes avant filtrage : 115106
Nombre de lignes après filtrage : 110889
Nombre de valeurs extrêmes : 4217

--- dvf_initial ---
Dimensions : (115106, 16)
Fichier enregistré : ../data/processed/dvf_initial.csv

--- dvf_nettoye ---
Dimensions : (110889, 16)
Fichier enregistré : ../data/processed/dvf_nettoye.csv


,Taux de NA (%) - initial,Taux de NA (%) - nettoyé
valeur_fonciere,0.00,0.00
code_commune,0.00,0.00
lot1_surface_carrez,58.16,57.41
nombre_lots,0.00,0.00
type_local,0.00,0.00
surface_reelle_bati,0.00,0.00
nombre_pieces_principales,0.10,0.09
surface_terrain,73.93,74.65
longitude,1.70,1.65
latitude,1.70,1.65


,Avant traitement,Après traitement
Nombre d'observations,115106.00,110889.00
Prix moyen au m²,4036.95,4031.80
Prix médian au m²,3860.47,3909.55
Écart-type,3136.26,1497.87
Prix minimum au m²,0.00,1148.55
Prix maximum au m²,361702.13,11428.57



Les deux datasets sont prêts pour la modélisation.
DVF initial avec prix extrêmes : (115106, 16)
DVF nettoyé sans prix extrêmes : (110889, 16)
Nombre de prix extrêmes retirés : 4217

Fichiers créés :
- ../data/processed/dvf_initial.csv
- ../data/processed/dvf_nettoye.csv
